### Learn PyTorch - Fundamentals 🔦

DOCS : https://docs.pytorch.org/tutorials/beginner/basics/quickstart_tutorial.html

In [1]:
# Imports
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [2]:
# Download training data from open datasets
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download the test data from open datasets
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.7MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 215kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 4.04MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 27.3MB/s]


In [3]:
batch_size = 64

# Create data loaders
train_dataloaders = DataLoader(training_data, batch_size=batch_size)
test_dataloaders = DataLoader(test_data, batch_size=batch_size)

for X,y in test_dataloaders:
   print(f"Shape of X [N, C, H, W] : {X.shape}")
   print(f"Shape of y : {y.shape} {y.dtype}")
   break

Shape of X [N, C, H, W] : torch.Size([64, 1, 28, 28])
Shape of y : torch.Size([64]) torch.int64


In [4]:
# Check if GPU exist for acceleration
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"

print(f"Using {device} device")

Using cpu device


In [14]:
# Define model
class NeuralNetwork(nn.Module):
  def __init__(self):
    super().__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
        nn.Linear(28*28, 512),
        nn.ReLU(),
        nn.Linear(512,512),
        nn.ReLU(),
        nn.Linear(512,10)
    )

  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits

In [15]:
model = NeuralNetwork().to(device)
print(model)

NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [16]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

In [17]:
def train(dataloader, model, loss_fn, optimizer):
  size = len(dataloader.dataset)
  model.train()
  for batch, (X,y) in enumerate(dataloader):
    X,y = X.to(device), y.to(device)

    #Compute prediction error
    pred = model(X)
    loss = loss_fn(pred, y)

    # Backpropagation
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if batch % 100 == 0:
      loss, current = loss.item(), (batch + 1) * len(X)
      print(f"loss : {loss:>7f} [{current:>5}/{size:>5d}]")

In [18]:
def test(dataloader, model, loss_fn):
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  model.eval()
  test_loss, correct = 0,0
  with torch.no_grad():
    for X,y in dataloader:
      X,y = X.to(device), y.to(device)
      pred = model(X)
      test_loss += loss_fn(pred, y).item()
      correct += (pred.argmax(1) == y).type(torch.float).sum().item()
  test_loss /= num_batches
  correct /= size
  print(f"Test Error: \n Accuracy : {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [19]:
epochs = 5
for t in range(epochs):
  print(f"Epoch {t+1}\n-----------------")
  train(train_dataloaders, model, loss_fn, optimizer)
  test(test_dataloaders, model, loss_fn)
print("Done!")

Epoch 1
-----------------
loss : 2.304801 [   64/60000]
loss : 2.284537 [ 6464/60000]
loss : 2.265766 [12864/60000]
loss : 2.256928 [19264/60000]
loss : 2.245203 [25664/60000]
loss : 2.216128 [32064/60000]
loss : 2.222695 [38464/60000]
loss : 2.183891 [44864/60000]
loss : 2.192274 [51264/60000]
loss : 2.141033 [57664/60000]
Test Error: 
 Accuracy : 50.5%, Avg loss: 2.141532 

Epoch 2
-----------------
loss : 2.164050 [   64/60000]
loss : 2.140326 [ 6464/60000]
loss : 2.080125 [12864/60000]
loss : 2.089758 [19264/60000]
loss : 2.035735 [25664/60000]
loss : 1.981127 [32064/60000]
loss : 2.005866 [38464/60000]
loss : 1.918067 [44864/60000]
loss : 1.940097 [51264/60000]
loss : 1.840450 [57664/60000]
Test Error: 
 Accuracy : 54.5%, Avg loss: 1.844258 

Epoch 3
-----------------
loss : 1.894484 [   64/60000]
loss : 1.845560 [ 6464/60000]
loss : 1.727819 [12864/60000]
loss : 1.763917 [19264/60000]
loss : 1.654367 [25664/60000]
loss : 1.613110 [32064/60000]
loss : 1.635336 [38464/60000]
loss :

In [20]:
# Save the model
#torch.save(model.state_dict(), "test_model.pth")

In [21]:
# Load model
#model = NeuralNetwork().to(device)
#model.load_state_dict(torch.load("test_model.pth", weights_only=True))

In [22]:
# Using model to make predicitons
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]

with torch.no_grad():
  x = x.to(device)
  pred = model(x)
  predicted, actual = classes[pred[0].argmax(0)], classes[y]
  print(f'Predicited: "{predicted}", Actual: "{actual}"')

Predicited: "Ankle boot", Actual: "Ankle boot"
